# Jasper 完整BF16工作站
先手动选择G4。此笔记本恢复已有完整模型，不下载量化替代，不自动关机。结束任务请回App备份核验后释放。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
CONTROL = {}
"""Runs only inside the explicitly selected VM. No unconditional auto-shutdown."""
from pathlib import Path
import hashlib,json,shutil,subprocess,sys,time,urllib.request,os,base64
ROOT=Path('/content/ComfyUI');WORK=Path('/content/jasper_workstation');WORK.mkdir(exist_ok=True)
BACKUP=Path('/content/drive/MyDrive/AI_Workstations/Jasper_Workstation')
H3=Path('/content/drive/MyDrive/AI_Workstations/MiniMax_H3_official')
def event(stage,message,**extra):print('JASPER_EVENT '+json.dumps(dict(stage=stage,message=message,**extra),ensure_ascii=False),flush=True)
def request(path,data=None):
 q=urllib.request.Request('http://127.0.0.1:8188'+path,data=None if data is None else json.dumps(data).encode(),headers={'Content-Type':'application/json'})
 with urllib.request.urlopen(q,timeout=15) as f:return json.load(f)
def digest(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(16*1024*1024),b''):h.update(b)
 return h.hexdigest()
def safe_child(root,rel):
 p=(root/rel).resolve()
 if not p.is_relative_to(root.resolve()):raise ValueError('清单包含越界路径')
 return p
def queue_idle():
 try:q=request('/queue')
 except Exception:
  # An unreachable backend may coexist with unrelated GPU work. Stop stays conservative.
  q=None
 if q and (q.get('queue_running') or q.get('queue_pending')):raise RuntimeError('ComfyUI 尚有运行或排队任务，请完成后备份/停止。')
 try:
  util=subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu','--format=csv,noheader,nounits'],text=True)
  if any(int(x.strip())>10 for x in util.splitlines() if x.strip()):raise RuntimeError('GPU 仍在工作，暂不执行备份后停止。')
 except FileNotFoundError:pass
 return q

def inspect():
 disk=shutil.disk_usage('/content');gpu=[]
 try:gpu=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,memory.used,utilization.gpu','--format=csv,noheader'],text=True).strip().splitlines()
 except Exception:pass
 try:stats=request('/system_stats');queue=request('/queue');comfy={'online':True,'queue_running':len(queue.get('queue_running',[])),'queue_pending':len(queue.get('queue_pending',[])),'stats':stats}
 except Exception as e:comfy={'online':False,'error':type(e).__name__}
 manifests=[]
 for p in sorted(H3.glob('models_manifest*.json')):
  d=json.loads(p.read_text());manifests.append({'name':p.name,'complete':d.get('complete',False),'files':len(d.get('files',[])),'bytes':sum(x.get('bytes',0) for x in d.get('files',[]))})
 models=[]
 for p in (ROOT/'models').rglob('*.safetensors') if (ROOT/'models').exists() else []:
  models.append({'file':str(p.relative_to(ROOT/'models')),'bytes':p.stat().st_size})
 return {'checked_at':time.time(),'gpu':gpu,'disk':{'total':disk.total,'free':disk.free,'used':disk.used},'drive_mounted':Path('/content/drive/MyDrive').is_dir(),'comfy':comfy,'models':models,'backup_manifests':manifests}

def restore():
 manifest=H3/'models_manifest_bf16.json'
 if not manifest.exists():raise RuntimeError('未发现完整 BF16 备份。请在官方笔记本挂载云盘，并核对 MiniMax_H3_official；不会改用量化版本。')
 d=json.loads(manifest.read_text())
 if not d.get('complete'):raise RuntimeError('BF16 备份清单未完成，停止恢复')
 files=d['files'];needed=sum(x['bytes'] for x in files if not safe_child(ROOT/'models',x['file']).exists())
 if shutil.disk_usage('/content').free<needed+8*2**30:raise RuntimeError('磁盘不足：需要模型文件空间及至少8GiB余量。')
 total=sum(x['bytes'] for x in files);done=0
 for item in files:
  src=safe_child(H3/'models',item['file']);dst=safe_child(ROOT/'models',item['file']);expected=item['sha256']
  event('verifying','校验 '+item['file'],bytes_done=done,bytes_total=total)
  if dst.exists() and dst.stat().st_size==item['bytes'] and digest(dst)==expected:done+=item['bytes'];continue
  if not src.exists() or src.stat().st_size!=item['bytes']:raise RuntimeError('云盘源文件缺失或长度不符：'+item['file'])
  dst.parent.mkdir(parents=True,exist_ok=True);part=dst.with_name(dst.name+'.partial');h=hashlib.sha256();copied=0;last=0
  with src.open('rb') as reader,part.open('wb') as writer:
   for block in iter(lambda:reader.read(16*1024*1024),b''):
    writer.write(block);h.update(block);copied+=len(block)
    if time.time()-last>2:event('restoring',item['file'],bytes_done=done+copied,bytes_total=total);last=time.time()
  if h.hexdigest()!=expected or copied!=item['bytes']:raise RuntimeError('恢复哈希不符：'+item['file'])
  part.replace(dst);done+=copied
 return {'manifest':str(manifest),'verified_files':len(files),'bytes':done,'precision':'BF16 full'}

def setup():
 import torch
 if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory<90*2**30:raise RuntimeError('未分配到目标96GB级G4，停止启动；不降级。')
 if not ROOT.exists():
  event('installing','下载官方 ComfyUI')
  subprocess.run(['git','clone','https://github.com/Comfy-Org/ComfyUI.git',str(ROOT)],check=True,stdout=sys.stderr,stderr=sys.stderr)
  subprocess.run(['git','-C',str(ROOT),'checkout','46659e65e95920e5d29cae011b43b02699c0bfd3'],check=True,stdout=sys.stderr,stderr=sys.stderr)
 try:
  info=request('/object_info')
  if 'MiniMaxH3ImageToVideo' in info:return {'online':True,'reused':True}
 except Exception:pass
 env=Path('/content/jasper_comfy_env');python=env/'bin/python'
 if not python.exists():subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(env)],check=True,stdout=sys.stderr,stderr=sys.stderr)
 event('installing','安装隔离环境依赖，保留系统环境')
 subprocess.run([str(python),'-m','pip','install','--no-cache-dir','-r',str(ROOT/'requirements.txt')],check=True,stdout=sys.stderr,stderr=sys.stderr)
 log=(WORK/'comfy.log').open('a');p=subprocess.Popen([str(python),str(ROOT/'main.py'),'--listen','127.0.0.1','--port','8188','--disable-auto-launch'],cwd=ROOT,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
 (WORK/'comfy_pid.json').write_text(json.dumps({'pid':p.pid}))
 for _ in range(120):
  try:
   info=request('/object_info')
   if 'MiniMaxH3ImageToVideo' not in info:raise RuntimeError('服务启动但未发现H3节点')
   return {'online':True,'pid':p.pid}
  except Exception:time.sleep(2)
 raise RuntimeError('ComfyUI未就绪，日志保存在 /content/jasper_workstation/comfy.log')

def backup(flush=False):
 queue_idle()
 if not Path('/content/drive/MyDrive').is_dir():raise RuntimeError('请先在官方笔记本授权挂载云盘。未备份，不会关机。')
 dest=BACKUP/'runs'/time.strftime('%Y%m%d_%H%M%S');dest.mkdir(parents=True,exist_ok=True);files=[]
 for label,folder in [('production',Path('/content/jasper_production')),('outputs',ROOT/'output'),('workflows',ROOT/'user/default/workflows'),('control',WORK)]:
  if folder.exists():files.extend((label,folder,p) for p in folder.rglob('*') if p.is_file() and not p.is_symlink())
 total=sum(p.stat().st_size for _,_,p in files);done=0;manifest=[]
 for label,folder,src in files:
  before=src.stat();target=dest/label/src.relative_to(folder);target.parent.mkdir(parents=True,exist_ok=True)
  event('backing_up',str(src.relative_to(folder)),bytes_done=done,bytes_total=total)
  part=target.with_name(target.name+'.partial');shutil.copy2(src,part)
  h=digest(src)
  if digest(part)!=h or src.stat().st_mtime_ns!=before.st_mtime_ns:raise RuntimeError('源文件仍在变化或备份校验失败；保留运行时。')
  part.replace(target);manifest.append({'file':str(target.relative_to(dest)),'bytes':before.st_size,'sha256':h});done+=before.st_size
 (dest/'manifest.json').write_text(json.dumps({'complete':True,'files':manifest,'created':time.time()},indent=2));event('saved','文件逐项校验完成，等待云盘刷新',bytes_done=done,bytes_total=total)
 if flush:
  from google.colab import drive
  drive.flush_and_unmount();event('flushed','云盘已完成刷新并卸载')
 return {'backup_path':str(dest),'files':len(manifest),'bytes':done,'hash_verified':True,'drive_flushed':flush,'models':'模型权重沿用独立持久备份，未重复复制'}

def generate(resume=False):
 config=CONTROL['config'];job=CONTROL['job'];out=WORK/'jobs'/job;out.mkdir(parents=True,exist_ok=True)
 if resume or (out/'submitted.json').exists():
  pid=CONTROL.get('prompt_id') or json.loads((out/'submitted.json').read_text())['prompt_id']
  event('resumed','恢复已有任务跟踪，不重新提交',prompt_id=pid)
 else:
  width,height={'480P':(480,848),'720P':(736,1280),'768P':(768,1360),'1080P':(1088,1920)}[config['resolution']]
  if config.get('ratio')=='16:9':width,height=height,width
  if config.get('ratio')=='1:1':width=height=min(width,height)
  ref=CONTROL['reference'];name='jasper_'+job+ref['suffix'];(ROOT/'input').mkdir(exist_ok=True);(ROOT/'input'/name).write_bytes(base64.b64decode(ref['base64']))
  required=[ROOT/'models/diffusion_models/minimax_h3_fl2va_pruned_bf16.safetensors',ROOT/'models/text_encoders/qwen3vl_32b_minimax_h3_bf16.safetensors',ROOT/'models/vae/minimax_h3_video_vae_fp16.safetensors']
  if any(not p.exists() for p in required):raise RuntimeError('完整BF16模型缺失，请先恢复模型。')
  def node(kind,**inputs):return {'class_type':kind,'inputs':inputs}
  frames=max(5,round(float(config['duration'])*24));frames+=(5-frames%17)%17
  graph={'1':node('UNETLoader',unet_name=required[0].name,weight_dtype='default'),'2':node('CLIPLoader',clip_name=required[1].name,type='minimax',device='default'),'3':node('VAELoader',vae_name=required[2].name),'5':node('LoadImage',image=name),'6':node('MiniMaxH3ImageToVideo',clip=['2',0],vae=['3',0],first_frame=['5',0],prompt=config['prompt'],width=width,height=height,length=frames),'8':node('BasicScheduler',model=['1',0],scheduler='simple',steps=20,denoise=1.0),'9':node('BasicGuider',model=['1',0],conditioning=['6',0]),'10':node('RandomNoise',noise_seed=int(config['seed'])),'11':node('KSamplerSelect',sampler_name='res_multistep'),'12':node('SamplerCustomAdvanced',noise=['10',0],guider=['9',0],sampler=['11',0],sigmas=['8',0],latent_image=['6',1]),'13':node('VAEDecode',samples=['12',0],vae=['3',0]),'15':node('CreateVideo',images=['13',0],fps=24,bit_depth=8),'16':node('SaveVideo',video=['15',0],filename_prefix='jasper/'+job,format='auto',codec='auto')}
  (out/'workflow_api.json').write_text(json.dumps(graph,indent=2));(out/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2))
  answer=request('/prompt',{'prompt':graph,'client_id':job});pid=answer['prompt_id'];(out/'submitted.json').write_text(json.dumps({'prompt_id':pid}));event('submitted','已提交真实推理任务',prompt_id=pid)
 started=time.time()
 while time.time()-started<21600:
  history=request('/history/'+pid)
  if pid in history:
   result=history[pid];(out/'history.json').write_text(json.dumps(result))
   if result.get('status',{}).get('status_str')=='error':raise RuntimeError(str(result['status'].get('messages',[]))[-1500:])
   outputs=[]
   for value in result.get('outputs',{}).values():
    for key in ('videos','gifs','images'):
     for item in value.get(key,[]):
      p=safe_child(ROOT/'output',str(Path(item.get('subfolder',''))/item['filename']))
      if p.exists() and p.suffix.lower() in ('.mp4','.webm','.png','.jpg','.jpeg') and str(p) not in outputs:outputs.append(str(p))
   if not outputs:raise RuntimeError('推理结束但没有实际输出文件')
   return {'prompt_id':pid,'outputs':outputs,'elapsed':time.time()-started}
  q=request('/queue');running=any(row[1]==pid for row in q.get('queue_running',[]));pending=any(row[1]==pid for row in q.get('queue_pending',[]))
  if not running and not pending:raise RuntimeError('任务既不在队列也没有结果；保留ID，请检查远端日志。')
  event('sampling' if running else 'waiting','采样运行中' if running else '等待队列',prompt_id=pid,elapsed=time.time()-started);time.sleep(5)
 raise TimeoutError('跟踪达到6小时；任务未自动取消，保留ID可恢复跟踪')


print(restore())

In [ ]:
print(setup())

In [ ]:
print(inspect())